In [1]:
from app.components.charts import (
    build_grouped_bar_chart,
    build_line_chart,
    build_pie_chart,
)

from infra.spark_utils import build_spark
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
spark = build_spark()

file_path = "../data/silver/snapshots/2026-03/2026-03_snapshot.csv"

df_silver = spark.read.csv(
    path = file_path,
    sep = ",", 
    header=True,
)

df_silver.show()

+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|          timestamp|data_apuracao| ano|mes_num|mes|instituicao_fin|              resumo|      tipo|                nome| qtd|preco_medio|preco_atual|aporte|    exposicao|
+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Stock | BERK34 | ...|     stock|              BERK34| 5.0|     120.02|     130.33|   0.0|internacional|
|2026-02-24 10:21:10|   2026-03-14|2026|      3|Mar|             XP|Renda Fixa | Teso...|renda fixa|             tesouro| 1.0|    24000.0|    30000.0|   0.0|     nacional|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | IVVB11 | ...|     stock|              IVVB11|15.0|     300.05|  

In [ ]:
# dfs graficos de linha de comparacao, de barra e valores atuais para botoes HOME
df_instituicao = (
    df_silver
    .groupBy("data_apuracao","ano", "mes", "instituicao_fin")
    .agg(F.sum(F.col("preco_atual")).alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("mes"),
        F.lit("INSTITUICAO").alias("tipo_escopo"),
        F.col("instituicao_fin"),
        F.lit("valor_total").alias("nome_metrica"),
        F.col("valor_total"),
    )
)

df_total = (
    df_silver
    .groupBy("data_apuracao", "ano", "mes")
    .agg(F.sum(F.col("preco_atual")).alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.col("mes"),
        F.lit("HOME").alias("tipo_escopo"),
        F.lit("ALL").alias("instituicao_fin"),
        F.lit("valor_total").alias("nome_metrica"),
        F.col("valor_total"),
    )
)

df_evolucao_ano_mes = df_instituicao.unionByName(df_total)
df_evolucao_ano_mes = df_evolucao_ano_mes.withColumn("valor_total", F.round(F.col("valor_total"), 2)) # garantia de 2 casas 

# grafico de linha 
df_evolucao_ano_mes.show()


latest_value_window = Window.partitionBy("instituicao_fin").orderBy(F.col("data_apuracao").desc())

df_val_atual = (
    df_evolucao_ano_mes
    .withColumn("rn", F.row_number().over(latest_value_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# botoes
df_val_atual.show()


df_evolucao_ano = (
    df_evolucao_ano_mes
    .groupBy("data_apuracao","ano", "instituicao_fin")
    .agg(F.sum("valor_total").alias("valor_total"))
    .select(
        F.col("data_apuracao"),
        F.col("ano"),
        F.lit("HOME").alias("tipo_escopo"),
        F.col("instituicao_fin"),
        F.lit("valor_total_anual").alias("nome_metrica"),
        F.round(F.col("valor_total"), 2).alias("valor_total")
    )
)

# grafico de barras
df_evolucao_ano.show()

+-------------+----+---+-----------+---------------+------------+-----------+
|data_apuracao| ano|mes|tipo_escopo|instituicao_fin|nome_metrica|valor_total|
+-------------+----+---+-----------+---------------+------------+-----------+
|   2026-03-14|2026|Mar|INSTITUICAO|             XP| valor_total|   30130.33|
|   2026-03-14|2026|Mar|INSTITUICAO|          CLEAR| valor_total|     702.18|
|   2026-03-14|2026|Mar|INSTITUICAO|         NUBANK| valor_total|     7600.0|
|   2026-03-14|2026|Mar|       HOME|            ALL| valor_total|   38432.51|
+-------------+----+---+-----------+---------------+------------+-----------+

+-------------+----+---+-----------+---------------+------------+-----------+
|data_apuracao| ano|mes|tipo_escopo|instituicao_fin|nome_metrica|valor_total|
+-------------+----+---+-----------+---------------+------------+-----------+
|   2026-03-14|2026|Mar|       HOME|            ALL| valor_total|   38432.51|
|   2026-03-14|2026|Mar|INSTITUICAO|          CLEAR| valor_tota

In [7]:
# dfs graficos de pizza HOME
latest_value_window_name = Window.partitionBy("instituicao_fin", "tipo", "nome").orderBy(F.col("data_apuracao").desc())

df_dados_atuais = (
    df_silver
    .withColumn("rn", F.row_number().over(latest_value_window_name))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_dados_atuais.show()

df_exposicao_all = (
    df_dados_atuais
    .groupBy('exposicao')
    .agg(F.sum(F.col("preco_atual")).alias("valor_total"))
    .select(
        F.lit("ALL").alias("instituicao_fin")
        , F.col('exposicao')
        , F.col("valor_total")
    )
)

df_exposicao_instituicao = (
    df_dados_atuais
    .groupBy('exposicao', 'instituicao_fin')
    .agg(F.sum(F.col("preco_atual")).alias("valor_total"))
    .select(
        F.col('instituicao_fin')
        , F.col('exposicao')
        , F.col("valor_total")
    )
)

df_exposicao = df_exposicao_all.unionByName(df_exposicao_instituicao)

# grafico pizza exposicao HOME e INSTITUTICAO
df_exposicao.show()

df_tipo_all = (
    df_dados_atuais
    .groupBy('tipo')
    .agg(F.sum(F.col("preco_atual")).alias("valor_total"))
    .select(
        F.lit("ALL").alias("instituicao_fin")
        , F.col('tipo')
        , F.col("valor_total")
    )
)

df_tipo_instituicao = (
    df_dados_atuais
    .groupBy('tipo', 'instituicao_fin')
    .agg(F.sum(F.col("preco_atual")).alias("valor_total"))
    .select(
        F.col('instituicao_fin')
        , F.col('tipo')
        , F.col("valor_total")
    )
)

df_tipo = df_tipo_all.unionByName(df_tipo_instituicao)

# grafico pizza tipo HOME e INSTITUTICAO
df_tipo.show()

+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|          timestamp|data_apuracao| ano|mes_num|mes|instituicao_fin|              resumo|      tipo|                nome| qtd|preco_medio|preco_atual|aporte|    exposicao|
+-------------------+-------------+----+-------+---+---------------+--------------------+----------+--------------------+----+-----------+-----------+------+-------------+
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | BERK34 | ...|     stock|              BERK34|10.0|     132.29|     130.33|   0.0|internacional|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock  | BOVA11 |...|     stock|              BOVA11|10.0|      125.2|     174.55|   0.0|     nacional|
|2026-02-24 10:22:11|   2026-03-14|2026|      3|Mar|          CLEAR|Stock | IVVB11 | ...|     stock|              IVVB11|15.0|     300.05|  

In [ ]:
df_val_atual_pd  = df_val_atual.toPandas()
df_val_atual_pd = df_val_atual_pd[df_val_atual_pd['instituicao_fin'] != 'ALL']

In [ ]:
pie_chart = build_pie_chart(df_val_atual_pd, "exposicao", "instituicao_fin", "valor_total")
pie_chart

In [ ]:
# line_chart = build_line_chart(df, title='evolution', x_col="mes", y_col="preco_atual", series_col='instituicao_fin')
# line_chart 

In [ ]:
# grouped_bar_chart = build_grouped_bar_chart(df, title='evolution', x_col="mes", y_col="preco_atual", series_col='instituicao_fin')
# grouped_bar_chart